# RAG Model with File Upload Support
### Upload your documents and ask questions with hallucination detection


- Upload **TXT / PDF / DOCX / CSV** files
- Chunk + index with **Dense (SentenceTransformers + FAISS)** and **Sparse (BM25)** retrievers
- Use a **Hybrid Retriever + CrossEncoder reranker**
- Generate answers (Flan-T5) **based on retrieved context**
- Run a lightweight **hallucination/grounding detector**



##  Section 1: Package Installation

In [1]:
!pip install pandas numpy torch transformers -q
!pip install sentence-transformers faiss-cpu rank-bm25 -q
!pip install datasets tqdm scikit-learn -q
!pip install nltk rouge-score -q
!pip install accelerate bitsandbytes -q
!pip install PyPDF2 python-docx -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 52.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.9 MB/s eta 0:00:00


## Section 2: Import Libraries

In [2]:
import pandas as pd
import numpy as np
import torch
import re
import warnings
from typing import List, Dict
from collections import Counter
from tqdm import tqdm
import os
import io

warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cpu


## 🔧 Section 3: NLTK Setup

In [3]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

##  Section 4: Document Loader

In [4]:
from google.colab import files
import PyPDF2
from docx import Document

class DocumentLoader:

    @staticmethod
    def load_txt(file_content):
        return file_content.decode('utf-8')

    @staticmethod
    def load_pdf(file_content):
        pdf_reader = PyPDF2.PdfReader(io.BytesIO(file_content))
        text = ""
        for page in pdf_reader.pages:
            text += page.extract_text() + "\n"
        return text

    @staticmethod
    def load_docx(file_content):
        doc = Document(io.BytesIO(file_content))
        text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
        return text

    @staticmethod
    def load_csv(file_content):
        df = pd.read_csv(io.BytesIO(file_content))
        text_columns = df.select_dtypes(include=['object']).columns
        texts = []
        for _, row in df.iterrows():
            row_text = " ".join([str(row[col]) for col in text_columns if pd.notna(row[col])])
            if row_text.strip():
                texts.append(row_text)
        return "\n\n".join(texts)

    @classmethod
    def load_document(cls, filename, file_content):
        ext = filename.lower().split('.')[-1]

        if ext == 'txt':
            return cls.load_txt(file_content)
        elif ext == 'pdf':
            return cls.load_pdf(file_content)
        elif ext == 'docx':
            return cls.load_docx(file_content)
        elif ext == 'csv':
            return cls.load_csv(file_content)
        else:
            raise ValueError(f"Unsupported file format: {ext}")

##  Section 5: File Upload Function

In [5]:
def upload_and_process_documents():
    print("Please upload your document(s)")
    print("Supported formats: TXT, PDF, DOCX, CSV")
    print("-" * 50)

    uploaded = files.upload()

    documents = []
    for filename, content in uploaded.items():
        try:
            print(f"\nProcessing: {filename}")
            text = DocumentLoader.load_document(filename, content)

            documents.append({
                'filename': filename,
                'text': text,
                'length': len(text)
            })
            print(f"✓ Loaded successfully ({len(text)} characters)")

        except Exception as e:
            print(f"✗ Error loading {filename}: {str(e)}")

    if not documents:
        raise ValueError("No documents were successfully loaded!")

    print(f"\n{'='*50}")
    print(f"Total documents loaded: {len(documents)}")
    print(f"Total characters: {sum(d['length'] for d in documents):,}")

    return documents

##  Section 6: Text Chunking

In [6]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 0) -> List[str]:
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start = end - overlap
    return chunks

def create_chunk_corpus(documents: List[Dict], chunk_size: int = 200, overlap_pct: float = 0.0) -> List[Dict]:
    overlap = int(chunk_size * overlap_pct)
    chunked_corpus = []

    for doc_idx, doc in enumerate(tqdm(documents, desc="Chunking documents")):
        doc_id = doc.get('filename', f'doc_{doc_idx}')

        for chunk_idx, chunk in enumerate(chunk_text(doc['text'], chunk_size, overlap)):
            if chunk.strip():
                chunked_corpus.append({
                    'doc_id': doc_id,
                    'chunk_id': f"{doc_id}_chunk_{chunk_idx}",
                    'text': chunk
                })

    return chunked_corpus

##  Section 7: Dense Retriever

In [7]:
from sentence_transformers import SentenceTransformer
import faiss

class DenseRetriever:
    def __init__(self, model_name='sentence-transformers/all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
        self.index = None
        self.corpus = None

    def index_corpus(self, corpus: List[Dict]):
        self.corpus = corpus
        texts = [doc['text'] for doc in corpus]
        embeddings = self.model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        faiss.normalize_L2(embeddings)
        self.index.add(embeddings)

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        query_emb = self.model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(query_emb)
        scores, indices = self.index.search(query_emb, top_k)
        return [{**self.corpus[idx], 'score': float(score), 'retrieval_method': 'dense'}
                for score, idx in zip(scores[0], indices[0])]

##  Section 8: Sparse Retriever (BM25)

In [8]:
from rank_bm25 import BM25Okapi

class SparseRetriever:
    def __init__(self):
        self.bm25 = None
        self.corpus = None

    def tokenize(self, text: str) -> List[str]:
        return re.findall(r'\b\w+\b', text.lower())

    def index_corpus(self, corpus: List[Dict]):
        self.corpus = corpus
        tokenized = [self.tokenize(doc['text']) for doc in tqdm(corpus, desc="Indexing BM25")]
        self.bm25 = BM25Okapi(tokenized)

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        scores = self.bm25.get_scores(self.tokenize(query))
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [{**self.corpus[idx], 'score': float(scores[idx]), 'retrieval_method': 'bm25'}
                for idx in top_idx]

##  Section 9: Hybrid Retriever with Reranking

In [9]:
from sentence_transformers import CrossEncoder

class HybridRetriever:
    def __init__(self, dense: DenseRetriever, sparse: SparseRetriever):
        self.dense = dense
        self.sparse = sparse
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    def retrieve(self, query: str, top_k: int = 5, initial_k: int = 20) -> List[Dict]:
        dense_res = self.dense.retrieve(query, initial_k)
        sparse_res = self.sparse.retrieve(query, initial_k)

        seen, candidates = set(), []
        for r in dense_res + sparse_res:
            if r['chunk_id'] not in seen:
                seen.add(r['chunk_id'])
                candidates.append(r)

        scores = self.reranker.predict([[query, c['text']] for c in candidates])
        for i, c in enumerate(candidates):
            c['rerank_score'] = float(scores[i])
            c['retrieval_method'] = 'hybrid'

        candidates.sort(key=lambda x: x['rerank_score'], reverse=True)
        return candidates[:top_k]

##  Section 10: Hallucination Detector

In [10]:
from sentence_transformers import util
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords

class HallucinationDetector:
    def __init__(self, embedding_model=None):
        self.embed_model = embedding_model or SentenceTransformer('all-MiniLM-L6-v2')
        self.stop_words = set(stopwords.words('english'))

    def check_semantic_grounding(self, answer: str, context_chunks: List[Dict], threshold: float = 0.5) -> Dict:
        context_text = " ".join([c['text'] for c in context_chunks])
        answer_emb = self.embed_model.encode(answer, convert_to_numpy=True)
        context_emb = self.embed_model.encode(context_text, convert_to_numpy=True)
        similarity = float(util.cos_sim(answer_emb, context_emb)[0][0])

        sentences = sent_tokenize(answer)
        sentence_scores = []
        for sent in sentences:
            if sent.strip():
                sent_emb = self.embed_model.encode(sent, convert_to_numpy=True)
                sim = float(util.cos_sim(sent_emb, context_emb)[0][0])
                sentence_scores.append({'sentence': sent, 'similarity': sim, 'grounded': sim >= threshold})

        ungrounded = [s for s in sentence_scores if not s['grounded']]
        return {
            'method': 'semantic_similarity',
            'overall_similarity': similarity,
            'is_hallucinated': similarity < threshold,
            'ungrounded_count': len(ungrounded),
            'total_sentences': len(sentence_scores)
        }

    def extract_entities(self, text: str) -> List[str]:
        capitalized = re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b', text)
        numbers = re.findall(r'\b\d+(?:\.\d+)?%?\b', text)
        return list(set(capitalized + numbers))

    def check_entity_grounding(self, answer: str, context_chunks: List[Dict]) -> Dict:
        context_text = " ".join([c['text'] for c in context_chunks]).lower()
        entities = self.extract_entities(answer)
        grounded = [e for e in entities if e.lower() in context_text]
        hallucinated = [e for e in entities if e.lower() not in context_text]
        ratio = len(grounded) / len(entities) if entities else 1.0
        return {
            'method': 'entity_verification',
            'total_entities': len(entities),
            'grounded_entities': grounded,
            'hallucinated_entities': hallucinated,
            'grounded_ratio': ratio,
            'is_hallucinated': ratio < 0.7
        }

    def check_ngram_overlap(self, answer: str, context_chunks: List[Dict], n: int = 3) -> Dict:
        def get_ngrams(text, n):
            words = re.findall(r'\b\w+\b', text.lower())
            return set(' '.join(words[i:i+n]) for i in range(len(words)-n+1))

        context_text = " ".join([c['text'] for c in context_chunks])
        answer_ng = get_ngrams(answer, n)
        context_ng = get_ngrams(context_text, n)

        if not answer_ng:
            return {'method': 'ngram_overlap', 'overlap_ratio': 1.0, 'is_hallucinated': False}

        overlap = answer_ng & context_ng
        ratio = len(overlap) / len(answer_ng)
        return {
            'method': 'ngram_overlap',
            'overlap_ratio': ratio,
            'is_hallucinated': ratio < 0.2
        }

    def check_claim_verification(self, answer: str, context_chunks: List[Dict], threshold: float = 0.6) -> Dict:
        claims = [s.strip() for s in sent_tokenize(answer) if '?' not in s and len(s.split()) > 4]
        verified, unverified = [], []

        for claim in claims:
            claim_emb = self.embed_model.encode(claim, convert_to_numpy=True)
            best_sim, best_match = 0.0, ""

            for chunk in context_chunks:
                for sent in sent_tokenize(chunk['text']):
                    if sent.strip():
                        sent_emb = self.embed_model.encode(sent, convert_to_numpy=True)
                        sim = float(util.cos_sim(claim_emb, sent_emb)[0][0])
                        if sim > best_sim:
                            best_sim, best_match = sim, sent

            result = {'claim': claim, 'similarity': best_sim, 'best_match': best_match, 'verified': best_sim >= threshold}
            (verified if result['verified'] else unverified).append(result)

        ratio = len(verified) / len(claims) if claims else 1.0
        return {
            'method': 'claim_verification',
            'total_claims': len(claims),
            'verified_claims': verified,
            'unverified_claims': unverified,
            'verification_ratio': ratio,
            'is_hallucinated': ratio < 0.7
        }

    def detect_hallucination(self, answer: str, context_chunks: List[Dict], query: str = None) -> Dict:
        checks = {
            'semantic': self.check_semantic_grounding(answer, context_chunks),
            'entity': self.check_entity_grounding(answer, context_chunks),
            'ngram': self.check_ngram_overlap(answer, context_chunks),
            'claims': self.check_claim_verification(answer, context_chunks)
        }

        scores = {
            'semantic': 1 - checks['semantic']['overall_similarity'],
            'entity': 1 - checks['entity']['grounded_ratio'],
            'ngram': 1 - checks['ngram']['overlap_ratio'],
            'claims': 1 - checks['claims']['verification_ratio']
        }
        weights = {'semantic': 0.35, 'entity': 0.15, 'ngram': 0.15, 'claims': 0.35}
        hallucination_score = sum(scores[k] * weights[k] for k in scores)

        if hallucination_score < 0.2:
            confidence = "HIGH - Well grounded"
        elif hallucination_score < 0.4:
            confidence = "MEDIUM - Mostly grounded"
        elif hallucination_score < 0.6:
            confidence = "LOW - Potential hallucinations"
        else:
            confidence = "VERY LOW - Likely hallucinated"

        issues = []
        if checks['semantic']['is_hallucinated']:
            issues.append(f"Low semantic similarity ({checks['semantic']['overall_similarity']:.2f})")
        if checks['entity']['hallucinated_entities']:
            issues.append(f"Ungrounded entities: {checks['entity']['hallucinated_entities'][:3]}")
        if checks['ngram']['is_hallucinated']:
            issues.append(f"Low n-gram overlap ({checks['ngram']['overlap_ratio']:.2f})")
        if checks['claims']['unverified_claims']:
            issues.append(f"{len(checks['claims']['unverified_claims'])} unverified claims")

        explanation = "Issues: " + "; ".join(issues) if issues else "Answer well-grounded in context."

        return {
            'hallucination_score': hallucination_score,
            'confidence_level': confidence,
            'is_likely_hallucinated': hallucination_score > 0.4,
            'explanation': explanation,
            'checks': checks
        }

##  Section 11: Answer Generator

In [11]:
from transformers import pipeline

class RAGGenerator:
    def __init__(self):
        self.pipe = pipeline("text2text-generation", model="google/flan-t5-base", max_length=256)

    def generate(self, query: str, context_chunks: List[Dict]) -> str:
        context = " ".join([c['text'] for c in context_chunks[:3]])[:1500]
        prompt = f"Answer based on context only. If unsure, say so.\n\nContext: {context}\n\nQuestion: {query}\n\nAnswer:"
        return self.pipe(prompt)[0]['generated_text']

##  Section 12: Complete RAG Pipeline

In [12]:
class RAGPipeline:
    def __init__(self, retriever, generator, detector, top_k=5):
        self.retriever = retriever
        self.generator = generator
        self.detector = detector
        self.top_k = top_k

    def query(self, question: str, detect_hallucinations: bool = True) -> Dict:
        retrieved = self.retriever.retrieve(question, self.top_k)
        answer = self.generator.generate(question, retrieved)

        result = {'question': question, 'answer': answer, 'retrieved_chunks': retrieved}

        if detect_hallucinations:
            hal_result = self.detector.detect_hallucination(answer, retrieved)
            result['hallucination'] = {
                'score': hal_result['hallucination_score'],
                'confidence': hal_result['confidence_level'],
                'is_hallucinated': hal_result['is_likely_hallucinated'],
                'explanation': hal_result['explanation']
            }

        return result

Section 13: System Initialization

In [13]:
def initialize_rag_system(documents, chunk_size=200, overlap_pct=0.0):

    print("\n" + "="*60)
    print("INITIALIZING RAG SYSTEM")
    print("="*60)

    print("\n[1/5] Creating chunks...")
    corpus = create_chunk_corpus(documents, chunk_size, overlap_pct)
    print(f"✓ Created {len(corpus)} chunks")

    print("\n[2/5] Initializing dense retriever...")
    dense_retriever = DenseRetriever()
    dense_retriever.index_corpus(corpus)
    print("✓ Dense retriever ready")

    print("\n[3/5] Initializing sparse retriever...")
    sparse_retriever = SparseRetriever()
    sparse_retriever.index_corpus(corpus)
    print("✓ Sparse retriever ready")

    print("\n[4/5] Creating hybrid retriever...")
    hybrid_retriever = HybridRetriever(dense_retriever, sparse_retriever)
    print("✓ Hybrid retriever ready")

    print("\n[5/5] Initializing generator and detector...")
    generator = RAGGenerator()
    detector = HallucinationDetector(embedding_model=dense_retriever.model)
    print("✓ All components ready")

    rag_pipeline = RAGPipeline(hybrid_retriever, generator, detector)

    print("\n" + "="*60)
    print("RAG SYSTEM READY!")
    print("="*60)

    return rag_pipeline

## Section 14: Interactive Query Session

In [14]:
def interactive_query_session(rag_pipeline):

    print("\n" + "="*60)
    print("INTERACTIVE QUERY SESSION")
    print("="*60)
    print("Type your questions (or 'quit' to exit)")
    print("-"*60)

    while True:
        query = input("\n Your question: ").strip()

        if query.lower() in ['quit', 'exit', 'q']:
            print("\nExiting session. Goodbye!")
            break

        if not query:
            continue

        try:
            print("\n🔍 Searching and generating answer...")
            result = rag_pipeline.query(query, detect_hallucinations=True)

            print("\n" + "-"*60)
            print(f"Answer: {result['answer']}")
            print(f"\n Hallucination Score: {result['hallucination']['score']:.3f}")
            print(f" Confidence: {result['hallucination']['confidence']}")
            print(f" {result['hallucination']['explanation']}")
            print("-"*60)

            print(f"\n Top Retrieved Sources:")
            for i, chunk in enumerate(result['retrieved_chunks'][:3], 1):
                score = chunk.get('rerank_score', chunk.get('score', 0))
                print(f"  [{i}] (score: {score:.3f}) {chunk['text'][:100]}...")

        except Exception as e:
            print(f"\n Error: {str(e)}")

## Section 15: Quick Start Function

In [15]:
def quick_start():
    documents = upload_and_process_documents()
    rag_pipeline = initialize_rag_system(documents)
    interactive_query_session(rag_pipeline)
    return rag_pipeline

print("""
╔══════════════════════════════════════════════════════════╗
║          RAG MODEL WITH FILE UPLOAD SUPPORT              ║
║          Upload your documents and ask questions!        ║
╚══════════════════════════════════════════════════════════╝

USAGE:
------
1. Run: documents = upload_and_process_documents()
2. Run: rag_pipeline = initialize_rag_system(documents)
3. Run: interactive_query_session(rag_pipeline)

OR use the quick start:
----------------------
Run: rag_pipeline = quick_start()
""")


╔══════════════════════════════════════════════════════════╗
║          RAG MODEL WITH FILE UPLOAD SUPPORT              ║
║          Upload your documents and ask questions!        ║
╚══════════════════════════════════════════════════════════╝

USAGE:
------
1. Run: documents = upload_and_process_documents()
2. Run: rag_pipeline = initialize_rag_system(documents)
3. Run: interactive_query_session(rag_pipeline)

OR use the quick start:
----------------------
Run: rag_pipeline = quick_start()



## 🎯 Section 16: Run the System
### Execute the cell below to start!

**Note:** This uses `google.colab.files.upload()` for uploads, so it’s designed for **Google Colab**.


In [ ]:
rag_pipeline = quick_start()

Please upload your document(s)
Supported formats: TXT, PDF, DOCX, CSV
--------------------------------------------------
